# SMART — Multi-Object Tracker
**S**tick via **M**otion **A**nd **R**ecognition **T**racker

Bu notebook üç ana işlemi kapsar:
1. **Kurulum** — Repo, bağımlılıklar, DCNv2 derleme
2. **Eğitim** — MOT17 / SOMPT22 üzerinde `tracking,embedding` görevi
3. **İnference** — Video üzerinde çoklu nesne takibi

> **Gereksinim:** Çalışma zamanı → GPU (T4 veya daha iyisi)

## 0 · GPU & CUDA Kontrolü

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.version.cuda)
print('GPU     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK — Runtime > GPU seç!')
assert torch.cuda.is_available(), 'GPU bulunamadı. Çalışma zamanını GPU olarak değiştir.'

## 1 · Google Drive Bağlantısı
Model ağırlıkları, dataset ve çıktılar Drive'a kaydedilir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/SMART'   # ← İstediğin klasörü değiştir
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive klasörü:', DRIVE_ROOT)

## 2 · Repo Kurulumu

In [ ]:
# Repo'yu klonla
%cd /content
!git clone https://github.com/sompt22/SMART.git
%cd /content/SMART

In [ ]:
# PyPI bağımlılıkları
!pip install -q \
    numpy==1.23.5 \
    opencv-python \
    Cython \
    numba \
    progress \
    matplotlib \
    easydict \
    scipy \
    pyyaml \
    yacs \
    cython-bbox \
    motmetrics \
    'lapx>=0.5.2' \
    openpyxl \
    Pillow \
    tensorboardX \
    fvcore \
    pycocotools \
    pyquaternion
print('Bağımlılıklar kuruldu.')

In [ ]:
# DCNv2 (Deformable Convolution) derle — DLA-34 için zorunlu
%cd /content/SMART/src/lib/model/networks/DCNv2
!rm -f *.so && rm -rf build/
!python setup.py build develop --user 2>&1 | tail -5
print('DCNv2 derlendi.')

In [ ]:
# External NMS Cython modülünü derle
%cd /content/SMART/src/lib/external
!python setup.py build_ext --inplace 2>&1 | tail -5
print('External NMS derlendi.')
%cd /content/SMART

In [ ]:
# sys.path'e SMART modüllerini ekle
import sys
sys.path.insert(0, '/content/SMART/src')
sys.path.insert(0, '/content/SMART/src/lib')
print('Path güncellendi.')

## 3 · Veri Seti Hazırlama

Aşağıdaki iki seçenekten birini çalıştır:
- **3A** → MOT17 (otomatik indir + COCO'ya dönüştür)
- **3B** → SOMPT22 / Özel Dataset (Drive'dan kopyala)

### 3A · MOT17 (otomatik)

In [ ]:
# ─── MOT17 ───────────────────────────────────────────────────────────────────
# Drive'da hazır zip varsa doğrudan kopyala, yoksa indir
import os

DATA_DIR     = '/content/SMART/data'
MOT17_DIR    = f'{DATA_DIR}/mot17'
MOT17_ZIP    = f'{DRIVE_ROOT}/MOT17.zip'   # Drive'daki zip yolu (opsiyonel)

os.makedirs(f'{MOT17_DIR}/images', exist_ok=True)
os.makedirs(f'{MOT17_DIR}/annotations', exist_ok=True)

if os.path.exists(MOT17_ZIP):
    print('Drive\'dan kopyalanıyor...')
    !cp "{MOT17_ZIP}" "{MOT17_DIR}/"
else:
    print('İndiriliyor (~4 GB, birkaç dakika sürebilir)...')
    !wget -q --show-progress -O "{MOT17_DIR}/MOT17.zip" \
        https://motchallenge.net/data/MOT17.zip

!unzip -q "{MOT17_DIR}/MOT17.zip" -d "{MOT17_DIR}/images/"
!mv "{MOT17_DIR}/images/MOT17/train" "{MOT17_DIR}/images/train" 2>/dev/null || true
!rm -f "{MOT17_DIR}/MOT17.zip"
print('MOT17 açıldı.')

In [ ]:
# MOT17 → COCO formatına dönüştür (train_half.json + val_half.json)
import sys, textwrap, importlib

CONVERT_SCRIPT = '/content/SMART/src/tools/mot17/convert_mot_to_coco.py'

# Script içindeki sabit DATA_PATH'i Colab path'iyle değiştir
with open(CONVERT_SCRIPT) as f:
    code = f.read()

code = code.replace(
    "DATA_PATH = '/home/fatih/phd/SMART/data/mot17/images/'",
    f"DATA_PATH = '{MOT17_DIR}/images/'"
).replace(
    "OUT_PATH = DATA_PATH + 'annotations/'",
    f"OUT_PATH = '{MOT17_DIR}/annotations/'"
)

exec(compile(code, CONVERT_SCRIPT, 'exec'))
print('Annotation dosyaları oluşturuldu:')
!ls -lh "{MOT17_DIR}/annotations/"

### 3B · SOMPT22 / Özel Dataset (Drive'dan)

In [ ]:
# ─── SOMPT22 ─────────────────────────────────────────────────────────────────
# Drive'da şu yapının hazır olduğu varsayılır:
#   MyDrive/SMART/sompt22/
#     images/train/   ← sekans klasörleri
#     images/val/
#     annotations/train.json
#     annotations/val.json

SOMPT22_SRC = f'{DRIVE_ROOT}/sompt22'
SOMPT22_DST = '/content/SMART/data/sompt22'

if os.path.exists(SOMPT22_SRC):
    !cp -r "{SOMPT22_SRC}" "{SOMPT22_DST}"
    print('SOMPT22 kopyalandı.')
    !ls -lh "{SOMPT22_DST}/annotations/"
else:
    print(f'SOMPT22 bulunamadı: {SOMPT22_SRC}')
    print('Drive\'a yükleyip yolu güncelle.')

### 3C · MOT formatından COCO'ya dönüştürme (genel)

In [ ]:
# SOMPT22 / özel dataset: MOT txt annotation → COCO JSON
# Önce convert_mot_to_coco.py scriptini kendi dataset yolunla çalıştır

CUSTOM_DATA_PATH = '/content/SMART/data/sompt22/images/'   # ← Değiştir
CUSTOM_OUT_PATH  = '/content/SMART/data/sompt22/annotations/'

%cd /content/SMART/src/tools/sompt22
!python convert_mot_to_coco.py 2>&1 | tail -10
%cd /content/SMART

## 4 · Ön-eğitimli Model
CenterNet DLA-34 ağırlıklarını başlangıç noktası olarak kullan.

In [ ]:
MODELS_DIR = '/content/SMART/models'
os.makedirs(MODELS_DIR, exist_ok=True)

# ── Seçenek A: Drive'daki mevcut .pth ────────────────────────────────────────
PRETRAIN_PATH = f'{DRIVE_ROOT}/ctdet_coco_dla_2x.pth'

if os.path.exists(PRETRAIN_PATH):
    !cp "{PRETRAIN_PATH}" "{MODELS_DIR}/"
    print('Model Drive\'dan kopyalandı.')
else:
    # ── Seçenek B: CenterNet resmi DLA-34 ağırlıklarını indir ─────────────────
    print('CenterNet DLA-34 COCO ağırlıkları indiriliyor...')
    !wget -q --show-progress -O "{MODELS_DIR}/ctdet_coco_dla_2x.pth" \
        https://dl.dropbox.com/s/shscf1k5qgkh3xm/ctdet_coco_dla_2x.pth
    PRETRAIN_PATH = f'{MODELS_DIR}/ctdet_coco_dla_2x.pth'

print('Ağırlık dosyası:', PRETRAIN_PATH)
!ls -lh "{MODELS_DIR}/"

## 5 · Eğitim Konfigürasyonu

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TÜM EĞİTİM PARAMETRELERİ — buradan düzenle
# ─────────────────────────────────────────────────────────────────────────────

CFG = dict(
    task         = 'tracking,embedding',  # görev
    dataset      = 'mot17',               # 'mot17' | 'mot20' | 'sompt22' | 'divo'
    exp_id       = 'colab_mot17_dla34',   # deney adı

    # Model
    arch         = 'dla_34',
    load_model   = f'{MODELS_DIR}/ctdet_coco_dla_2x.pth',

    # Eğitim
    num_epochs   = 30,
    lr           = '1e-4',
    lr_step      = '20,25',
    batch_size   = 8,           # T4: 8, A100: 16
    num_workers  = 2,
    optim        = 'adam',      # 'adam' | 'adamw' | 'sgd'

    # Loss
    multi_loss       = 'uncertainty',
    embedding_loss   = 'ce',    # 'ce' | 'focal'
    embedding_weight = 1.0,
    know_dist_weight = 0.0,     # 0 = KD kapalı

    # Augmentation
    hm_disturb   = 0.05,
    lost_disturb = 0.4,
    fp_disturb   = 0.1,

    # Kayıt
    val_intervals = 5,
    save_dir      = f'{DRIVE_ROOT}/exp',  # Drive'a yedekle
    gpus          = '0',
)

print('Konfigürasyon:')
for k, v in CFG.items():
    print(f'  {k:20s}: {v}')

## 6 · Eğitim

In [ ]:
# Eğitim komutunu oluştur
FREEZE = ''''{"base":false,"dla_up":false,"ida_up":false,
             "hm":false,"reg":false,"wh":false,
             "ltrb_amodal":false,"embedding":false,"tracking":false}' '''

TRAIN_CMD = f"""
python /content/SMART/src/main.py {CFG['task']} \\
  --exp_id          {CFG['exp_id']} \\
  --dataset         {CFG['dataset']} \\
  --data_dir        /content/SMART/data \\
  --save_dir        {CFG['save_dir']} \\
  --arch            {CFG['arch']} \\
  --load_model      {CFG['load_model']} \\
  --num_epochs      {CFG['num_epochs']} \\
  --lr              {CFG['lr']} \\
  --lr_step         {CFG['lr_step']} \\
  --batch_size      {CFG['batch_size']} \\
  --num_workers     {CFG['num_workers']} \\
  --optim           {CFG['optim']} \\
  --multi_loss      {CFG['multi_loss']} \\
  --embedding_loss  {CFG['embedding_loss']} \\
  --embedding_weight {CFG['embedding_weight']} \\
  --know_dist_weight {CFG['know_dist_weight']} \\
  --hm_disturb      {CFG['hm_disturb']} \\
  --lost_disturb    {CFG['lost_disturb']} \\
  --fp_disturb      {CFG['fp_disturb']} \\
  --val_intervals   {CFG['val_intervals']} \\
  --gpus            {CFG['gpus']} \\
  --num_classes     1 \\
  --ltrb_amodal \\
  --pre_hm \\
  --same_aug \\
  --freeze_components {FREEZE.strip()}
""".strip()

print(TRAIN_CMD)

In [ ]:
# Eğitimi başlat
import subprocess
proc = subprocess.Popen(
    TRAIN_CMD, shell=True, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\nEğitim tamamlandı. Return code:', proc.returncode)

In [ ]:
# Eğitimi kaldığı yerden devam ettir (--resume)
# Bitmemiş veya uzun eğitimlerde bu hücreyi çalıştır

RESUME_CMD = TRAIN_CMD + ' --resume'

proc = subprocess.Popen(
    RESUME_CMD, shell=True, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\nResume tamamlandı. Return code:', proc.returncode)

## 7 · TensorBoard ile Eğitim Takibi

In [ ]:
%load_ext tensorboard
LOG_DIR = f"{CFG['save_dir']}/{CFG['task']}/{CFG['exp_id']}"
%tensorboard --logdir "{LOG_DIR}"

## 8 · İnference (Video Üzerinde MOT)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# İNFERENCE KONFİGÜRASYONU
# ─────────────────────────────────────────────────────────────────────────────

# Model: eğitimden çıkan son model ya da Drive'daki hazır model
INFER_MODEL = f"{CFG['save_dir']}/{CFG['task']}/{CFG['exp_id']}/model_last.pth"

# Yoksa Drive'dan kullan:
# INFER_MODEL = f'{DRIVE_ROOT}/model_last.pth'

# Video: Drive'dan kopyala
VIDEO_SRC = f'{DRIVE_ROOT}/test_video.mp4'  # ← Drive'daki video yolu
VIDEO_DST = '/content/test_video.mp4'

if os.path.exists(VIDEO_SRC):
    !cp "{VIDEO_SRC}" "{VIDEO_DST}"
    print('Video kopyalandı:', VIDEO_DST)
else:
    # Yoksa örnek MOT17 sekansından kısa clip oluştur
    SEQ = f'{MOT17_DIR}/images/train/MOT17-02-FRCNN/img1'
    if os.path.exists(SEQ):
        !ffmpeg -q -framerate 30 -pattern_type glob \
            -i "{SEQ}/*.jpg" -c:v libx264 -t 10 "{VIDEO_DST}" 2>/dev/null
        print('MOT17 görüntülerinden video oluşturuldu:', VIDEO_DST)
    else:
        print('Video bulunamadı. VIDEO_SRC yolunu güncelle.')

DEMO_EXP = 'colab_inference'
DEMO_OUT  = f'/content/SMART/exp/{CFG["task"]}/{DEMO_EXP}'
print('Çıktı klasörü:', DEMO_OUT)

In [ ]:
# İnference komutunu çalıştır
DEMO_CMD = f"""
python /content/SMART/src/demo.py {CFG['task']} \\
  --exp_id      {DEMO_EXP} \\
  --load_model  {INFER_MODEL} \\
  --demo        {VIDEO_DST} \\
  --num_classes 1 \\
  --gpus        0 \\
  --ltrb_amodal \\
  --track_thresh 0.4 \\
  --pre_thresh   0.5 \\
  --max_age      50 \\
  --save_video \\
  --no_pause \\
  --debug       0
""".strip()

print(DEMO_CMD)
proc = subprocess.Popen(
    DEMO_CMD, shell=True, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\nİnference tamamlandı. Return code:', proc.returncode)

In [ ]:
# Çıktı videosunu göster
from IPython.display import Video
import glob

output_videos = glob.glob(f'{DEMO_OUT}/**/*.avi', recursive=True) + \
                glob.glob(f'{DEMO_OUT}/**/*.mp4', recursive=True)

if output_videos:
    OUT_VIDEO = output_videos[0]
    # Colab için mp4'e çevir
    OUT_MP4 = OUT_VIDEO.replace('.avi', '_out.mp4')
    !ffmpeg -q -i "{OUT_VIDEO}" -vcodec libx264 "{OUT_MP4}" 2>/dev/null
    display(Video(OUT_MP4, embed=True, width=800))
    print('Video:', OUT_MP4)

    # Drive'a kaydet
    !cp "{OUT_MP4}" "{DRIVE_ROOT}/"
    print('Drive\'a kaydedildi.')
else:
    print('Video bulunamadı:', DEMO_OUT)
    !ls -lR "{DEMO_OUT}" 2>/dev/null || echo 'Klasör yok.'

## 9 · Değerlendirme (MOT Metrikleri)

In [ ]:
# HOTA / MOTA / IDF1 metriklerini hesapla
EVAL_CMD = f"""
python /content/SMART/src/test.py {CFG['task']} \\
  --exp_id      colab_eval \\
  --dataset     {CFG['dataset']} \\
  --data_dir    /content/SMART/data \\
  --load_model  {INFER_MODEL} \\
  --num_classes 1 \\
  --gpus        0 \\
  --ltrb_amodal \\
  --track_thresh 0.4 \\
  --pre_thresh   0.5 \\
  --max_age      150 \\
  --trainval
""".strip()

print(EVAL_CMD)
proc = subprocess.Popen(
    EVAL_CMD, shell=True, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\nDeğerlendirme tamamlandı.')

## 10 · Modeli Drive'a Yedekle

In [ ]:
EXP_DIR = f"{CFG['save_dir']}/{CFG['task']}/{CFG['exp_id']}"

if os.path.exists(EXP_DIR):
    # Sadece .pth dosyalarını kopyala
    BACKUP_DIR = f'{DRIVE_ROOT}/checkpoints/{CFG["exp_id"]}'
    os.makedirs(BACKUP_DIR, exist_ok=True)
    !cp "{EXP_DIR}"/*.pth "{BACKUP_DIR}/" 2>/dev/null
    !cp "{EXP_DIR}"/*.log "{BACKUP_DIR}/" 2>/dev/null
    print('Yedekleme tamamlandı:')
    !ls -lh "{BACKUP_DIR}/"
else:
    print('Deney klasörü bulunamadı:', EXP_DIR)

## 11 · Tek Kare Üzerinde Görselleştirme

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch, sys

sys.path.insert(0, '/content/SMART/src')
sys.path.insert(0, '/content/SMART/src/lib')

from opts import opts
from detector import Detector

# Argümanları doğrudan opts'a ver
opt = opts().parse([
    CFG['task'],
    '--load_model',  INFER_MODEL,
    '--dataset',     CFG['dataset'],
    '--num_classes', '1',
    '--gpus',        '0',
    '--ltrb_amodal',
    '--track_thresh','0.4',
    '--pre_thresh',  '0.5',
    '--max_age',     '50',
    '--no_pause',
    '--debug',       '0',
])

detector = Detector(opt)

# Test görüntüsü yükle
TEST_IMG = sorted(glob.glob(f'{MOT17_DIR}/images/train/MOT17-02-FRCNN/img1/*.jpg'))[0]
img = cv2.imread(TEST_IMG)

ret = detector.run(img)
results = ret['results']
print(f'Tespit: {len(results)} nesne | Toplam süre: {ret["tot"]:.3f}s')

# Sonuçları çiz
vis = img.copy()
for det in results:
    x1,y1,x2,y2 = [int(v) for v in det['bbox']]
    tid = det.get('tracking_id', -1)
    sc  = det['score']
    color = tuple(int(c) for c in plt.cm.hsv((tid * 37) % 256 / 256)[:3] * 255)
    cv2.rectangle(vis, (x1,y1), (x2,y2), color, 2)
    cv2.putText(vis, f'ID:{tid} {sc:.2f}', (x1, y1-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f'{len(results)} detections')
plt.show()